# 2. VectorRAG Pipeline — Medical Textbook AlzRAGBench

**Part of the AlzRAGBench project** — Medical Textbook variant.

This notebook implements **VectorRAG** over a 31-document Alzheimer's corpus featuring a **StatPearls medical textbook chapter**, PubMed abstracts, and Wikipedia articles. Chunks are embedded with `all-MiniLM-L6-v2` and indexed using FAISS L2.

## 2.1 Imports and Paths

In [ ]:
import os
import json
import pickle
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Paths
NOTEBOOK_DIR = Path(".").resolve()
BASE_DIR     = NOTEBOOK_DIR.parent
DATASET_DIR  = BASE_DIR / "Dataset"
CHUNKS_PATH  = DATASET_DIR / "chunking" / "_all_chunks.json"
VECTOR_DIR   = BASE_DIR / "vector_output"
VECTOR_DIR.mkdir(exist_ok=True)

print(f"Chunks dataset exists: {CHUNKS_PATH.exists()}")

## 2.2 Load Textbook & Article Chunks

In [ ]:
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [c["text"] for c in chunks]
metadata = [
    {
        "chunk_id": c["chunk_id"],
        "article_id": c["article_id"],
        "title": c["metadata"].get("title", "")
    }
    for c in chunks
]

print(f"Total Chunks Loaded: {len(texts)}")

# Source breakdown
from collections import Counter
sources = Counter([m["article_id"].split("_")[0] for m in metadata])
print(f"Chunk Source Distribution: {dict(sources)}")

## 2.3 Build FAISS Embedding Index

In [ ]:
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")

INDEX_FILE = VECTOR_DIR / "faiss.index"
TEXT_FILE  = VECTOR_DIR / "chunk_texts.pkl"
META_FILE  = VECTOR_DIR / "metadata.pkl"

if INDEX_FILE.exists():
    faiss_index = faiss.read_index(str(INDEX_FILE))
    with open(TEXT_FILE, "rb") as f: texts = pickle.load(f)
    with open(META_FILE, "rb") as f: metadata = pickle.load(f)
    print(f"Loaded existing FAISS index: {faiss_index.ntotal} vectors")
else:
    print("Encoding chunks into FAISS index...")
    embeddings = EMBED_MODEL.encode(texts, show_progress_bar=True, convert_to_numpy=True).astype("float32")
    faiss_index = faiss.IndexFlatL2(embeddings.shape[1])
    faiss_index.add(embeddings)
    faiss.write_index(faiss_index, str(INDEX_FILE))
    with open(TEXT_FILE, "wb") as f: pickle.dump(texts, f)
    with open(META_FILE, "wb") as f: pickle.dump(metadata, f)
    print(f"Indexed {faiss_index.ntotal} vectors successfully.")

## 2.4 VectorRAG Retrieval Function

In [ ]:
def vector_retrieve(query, top_k=5):
    """Retrieve top-k chunks by L2 distance."""
    q_emb = EMBED_MODEL.encode([query], convert_to_numpy=True).astype("float32")
    dists, idxs = faiss_index.search(q_emb, top_k)
    results = []
    for rank, idx in enumerate(idxs[0]):
        if idx != -1:
            results.append({
                "rank": rank + 1,
                "score": float(dists[0][rank]),
                "text": texts[idx],
                "metadata": metadata[idx]
            })
    return results

# Test Vector Retrieval
res = vector_retrieve("What are the main risk factors for Alzheimer's disease?", top_k=2)
for r in res:
    print(f"Rank {r['rank']} | Score: {r['score']:.4f} | Source: {r['metadata']['article_id']}")
    print(f"{r['text'][:150]}...\n")